In [2]:
!pip install transformers datasets scikit-learn streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 82.6 MB/s eta 0:00:00


In [3]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

In [4]:
dataset = load_dataset("ag_news")
label_names = dataset["train"].features["label"].names
print("Labels:", label_names)
print("Sample:", dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Labels: ['World', 'Sports', 'Business', 'Sci/Tech']
Sample: {'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [6]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [8]:
training_args = TrainingArguments(
    output_dir="./bert-ag-news",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
# Balanced sample: 2500 per class = 10,000 total (manageable + accurate)
def balanced_sample(dataset, per_class=2500):
    indices = []
    for label in range(4):
        class_indices = [i for i, x in enumerate(dataset) if x["label"] == label]
        indices += class_indices[:per_class]
    return dataset.select(indices)

train_dataset = balanced_sample(tokenized["train"], per_class=2500)
eval_dataset  = tokenized["test"].select(range(2000))

print(f"Train size: {len(train_dataset)}")
print(f"Eval size:  {len(eval_dataset)}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Train size: 10000
Eval size:  2000


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.275245,0.252467,0.923000,0.922832
2,0.189214,0.250167,0.927500,0.927534
3,0.096960,0.288041,0.925000,0.924763


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1875, training_loss=0.2688740778605143, metrics={'train_runtime': 705.87, 'train_samples_per_second': 42.501, 'train_steps_per_second': 2.656, 'total_flos': 1973368350720000.0, 'train_loss': 0.2688740778605143, 'epoch': 3.0})

In [10]:
results = trainer.evaluate()
print(f"\n✅ Accuracy : {results['eval_accuracy']:.4f}")
print(f"✅ F1 Score : {results['eval_f1']:.4f}")


✅ Accuracy : 0.9280
✅ F1 Score : 0.9280


In [11]:
trainer.save_model("./bert-ag-news")
tokenizer.save_pretrained("./bert-ag-news")
print("✅ Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


In [12]:
app_code = '''
import streamlit as st
import torch
from transformers import BertTokenizer, BertForSequenceClassification

@st.cache_resource
def load_model():
    tokenizer = BertTokenizer.from_pretrained("./bert-ag-news")
    model = BertForSequenceClassification.from_pretrained("./bert-ag-news")
    model.eval()
    return tokenizer, model

label_names = ["World", "Sports", "Business", "Sci/Tech"]
label_icons = ["🌍", "⚽", "💼", "🔬"]
label_colors = ["#4A90D9", "#27AE60", "#E67E22", "#8E44AD"]

st.set_page_config(page_title="News Classifier", page_icon="📰", layout="centered")

st.markdown("""
    <style>
        .main { background-color: #f9f9f9; }
        .result-box {
            padding: 1.2rem 1.5rem;
            border-radius: 12px;
            background: #ffffff;
            border-left: 6px solid;
            box-shadow: 0 2px 10px rgba(0,0,0,0.08);
            margin-top: 1rem;
        }
    </style>
""", unsafe_allow_html=True)

st.markdown("# 📰 News Topic Classifier")
st.markdown("Fine-tuned BERT on AG News — World · Sports · Business · Sci/Tech")
st.divider()

tokenizer, model = load_model()

headline = st.text_input("Enter a news headline:", placeholder="e.g. Tesla stock surges after record earnings")

if st.button("🔍 Classify", use_container_width=True):
    if not headline.strip():
        st.warning("Please enter a headline first.")
    else:
        with st.spinner("Analyzing..."):
            inputs = tokenizer(
                headline,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            )
            with torch.no_grad():
                outputs = model(**inputs)

            probs = torch.softmax(outputs.logits, dim=-1)[0]
            pred_id = torch.argmax(probs).item()
            confidence = probs[pred_id].item()

        # ✅ Use st. functions directly — no f-string HTML needed
        st.success(f"{label_icons[pred_id]}  {label_names[pred_id]}  —  {confidence*100:.1f}% confidence")

        st.markdown("#### All Category Scores")
        for i in range(4):
            st.progress(
                float(probs[i]),
                text=f"{label_icons[i]} {label_names[i]}: {probs[i]*100:.1f}%"
            )

st.divider()
st.caption("Model: bert-base-uncased | Dataset: AG News | Balanced 10k training samples")
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written!")

✅ app.py written!


In [13]:
!pip install pyngrok -q

from pyngrok import ngrok
import subprocess, time

# ✅ Paste your token from https://dashboard.ngrok.com
ngrok.set_auth_token("3DX1sVRYMH5BlVPBCm8JpGdjrGr_56oAjMXXruQLt8LVFZiQ3")

# Kill old processes
!pkill -f streamlit 2>/dev/null
!pkill -f ngrok    2>/dev/null
time.sleep(2)

# Start streamlit
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(5)

# Open tunnel
url = ngrok.connect(8501, bind_tls=True)
print(f"\n✅ App is live at: {url.public_url}")
print("👆 Click the link above to open your app!")

^C
^C

✅ App is live at: https://cameo-scooter-distill.ngrok-free.dev
👆 Click the link above to open your app!
